In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║         PDF → Markdown  |  Powered by Baidu Unlimited-OCR       ║
# ║         Google Colab Compatible  •  GPU Runtime Required         ║
# ║  GitHub: https://github.com/baidu/Unlimited-OCR                  ║
# ╚══════════════════════════════════════════════════════════════════╝
#
## ⚠️ Important

# Before running the script, go to Colab Runtime → Change runtime type and select T4 GPU or A100.

# How to Use

# Follow these 3 simple steps:

# 1. Upload the script to Google Colab or copy the code into Colab cells.
# 2. Set the PDF_PATH variable to the path of your PDF file.
# 3. Run all cells. The output .md file will be generated automatically and downloaded.

# ══════════════════════════════════════════════════════════════════

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 ▸ GPU CHECK
# ─────────────────────────────────────────────────────────────────

import subprocess, sys

def check_gpu():
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            gpu_info = result.stdout.strip()
            print(f"✅ GPU detected: {gpu_info}")
            return True
        else:
            print("❌ GPU Not Detected! Please open Runtime → Change runtime type → Hardware accelerator, then select T4 GPU and try again.")
            return False
    except Exception as e:
        print(f"❌ nvidia-smi error: {e}")
        return False

gpu_ok = check_gpu()
if not gpu_ok:
    raise SystemExit("GPU Required! Please change the runtime type to a GPU-enabled runtime and run the process again.")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 ▸ INSTALL DEPENDENCIES
# ─────────────────────────────────────────────────────────────────
#
# NOTE: This cell takes ~2–3 minutes to complete. Output is suppressed, so please wait.
#
# If you want to run it manually, paste the following in the terminal:
# pymupdf==1.27.2.2 psutil==7.2.2 -q
#
# NOTE: Pillow 12.1.1 (and 12.0.0) has a known broken wheel issue.
# PIL/ImageText.py imports `_Ink`, which does not exist in PIL/_typing.py,
# causing torchvision imports to fail.
#
# Therefore, Pillow is pinned to 11.3.0 (the last stable 11.x release),
# which does not contain this buggy module.
#
# ─────────────────────────────────────────────────────────────────

print("📦 Installing dependencies... (2-3 min)")

packages = [
    "torch==2.10.0",
    "torchvision==0.25.0",
    "transformers==4.57.1",
    "Pillow==11.3.0",
    "einops==0.8.2",
    "addict==2.4.0",
    "easydict==1.13",
    "pymupdf==1.27.2.2",
    "psutil==7.2.2",
]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet"] + packages,
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✅ All packages installed successfully!")
else:
    print("⚠️  Some packages may have warnings:")
    print(result.stderr[-1000:] if result.stderr else "")
    print("Continuing anyway...")

print(
    "\n⚠️ IMPORTANT: If you previously ran the old Pillow==12.1.1 version and imported PIL/torch, restart the Colab session (Runtime → Restart session) and run again from Cell 1; otherwise, the old broken PIL module may remain cached in memory."
)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 ▸ IMPORTS
# ─────────────────────────────────────────────────────────────────

import os
import re
import glob
import tempfile
import shutil
from pathlib import Path

import torch
import fitz  # PyMuPDF

print("✅ Core imports done.")
print(f"   PyTorch version : {torch.__version__}")
print(f"   CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU             : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   VRAM            : {vram:.1f} GB")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 ▸ LOAD MODEL  (The first run may take 5–15 minutes because the ~7 GB model needs to be downloaded.)
# ─────────────────────────────────────────────────────────────────

from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "baidu/Unlimited-OCR"

print(f"\n🤖 Loading model: {MODEL_NAME}")
print("   (First run: ~7 GB download — please be patient)\n")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)
print("✅ Tokenizer loaded.")

model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
)
model = model.eval().cuda()
print("✅ Model loaded & moved to GPU!\n")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 ▸ HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────


def pdf_to_images(pdf_path: str, dpi: int = 300) -> list:
    """
    Convert each page of the PDF to a PNG image at the specified DPI.
    """
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    tmp_dir = tempfile.mkdtemp(prefix="pdf_pages_")
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    paths = []

    print(f"   📄 PDF pages     : {total_pages}")
    print(f"   🖼️  DPI           : {dpi}")
    print(f"   📁 Temp dir      : {tmp_dir}")
    print()

    for i, page in enumerate(doc):
        out_path = os.path.join(tmp_dir, f"page_{i + 1:04d}.png")
        page.get_pixmap(matrix=mat).save(out_path)
        paths.append(out_path)
        done = int(30 * (i + 1) / total_pages)
        bar = "█" * done + "░" * (30 - done)
        print(f"\r   [{bar}] {i+1}/{total_pages}", end="", flush=True)

    doc.close()
    print(f"\n   ✅ All {total_pages} pages converted to images.\n")
    return paths


# Clean the <|det|>...</|det|> markers from the OCR output.
_DET_RE = re.compile(
    r"<\|det\|>([^<\s]+)(?:\s*\[[^\]]*\])?\s*<\|/det\|>(.*)",
    re.DOTALL,
)

def remove_det(raw: str) -> str:
    """
    Strip <|det|>type [bbox]<|/det|> markers.
    Ignore all images and convert only the existing text into Markdown.
    """
    blocks = []
    cur = None
    for line in raw.splitlines():
        line = line.rstrip()
        if not line:
            continue
        m = _DET_RE.match(line)
        if m:
            category = m.group(1).strip()
            content = m.group(2).strip()
            if category == "image":
                continue
            if cur is not None:
                blocks.append(cur)
            cur = [content] if content else []
            continue
        if cur is None:
            cur = []
        cur.append(line)
    if cur is not None:
        blocks.append(cur)
    return "\n\n".join("\n".join(b) for b in blocks).strip()


# ★ NEW: Remove broken image Markdown links (images are not saved, so the links will not work).
_IMG_MD_RE = re.compile(r"!\[[^\]]*\]\([^\)]*\)")

def strip_broken_images(text: str) -> str:
    """
    Strip broken image Markdown links (images are not saved, so the links will not work).
    """
    lines = text.splitlines()
    out = []
    for line in lines:
        if _IMG_MD_RE.search(line.strip()):
            # Leave a placeholder note, so it’s clear there was an image here
            out.append("*[Image on this page — not extracted]*")
        else:
            out.append(line)
    return "\n".join(out)


# ★ NEW: The model may sometimes enter a degenerate state and repeat the same phrase multiple times
# (e.g. "Respond to client only with data needed" 20+ times).
# This function detects and trims consecutive near-duplicate lines.
_BULLET_PREFIX_RE = re.compile(r"^[\s○●•\-\*]+")

def collapse_repetition(text: str, prefix_len: int = 10, max_repeat: int = 2) -> str:
    lines = text.split("\n")
    out = []
    i, n = 0, len(lines)
    while i < n:
        norm = _BULLET_PREFIX_RE.sub("", lines[i]).strip().lower()
        prefix = norm[:prefix_len]

        if not prefix:
            out.append(lines[i])
            i += 1
            continue

        # Count how many consecutive lines share the same prefix.
        j = i + 1
        while j < n:
            nxt = _BULLET_PREFIX_RE.sub("", lines[j]).strip().lower()
            if nxt[:prefix_len] == prefix:
                j += 1
            else:
                break

        run_len = j - i
        if run_len > max_repeat:
            out.extend(lines[i:i + max_repeat])
            out.append(f"*…[{run_len - max_repeat} repeated/degenerate lines trimmed]…*")
            i = j
        else:
            out.append(lines[i])
            i += 1

    return "\n".join(out)


def read_and_combine_outputs(output_dir: str, pdf_stem: str) -> str:
    """
    Read all Markdown files from the OCR output directory and combine them,
    applying repetition and broken-image cleanup.
    """
    md_files = sorted(glob.glob(os.path.join(output_dir, "*.md")))

    if not md_files:
        md_files = sorted(glob.glob(os.path.join(output_dir, "*.txt")))

    if not md_files:
        all_files = [
            os.path.join(output_dir, f)
            for f in sorted(os.listdir(output_dir))
            if os.path.isfile(os.path.join(output_dir, f))
        ]
        md_files = all_files

    if not md_files:
        print("   ⚠️  Warning: No output files found in:", output_dir)
        print("   Files present:", os.listdir(output_dir))
        return ""

    print(f"   📑 Found {len(md_files)} output file(s)")

    pages_text = []
    for i, fpath in enumerate(md_files):
        try:
            with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                raw = f.read()
            cleaned = remove_det(raw)
            cleaned = strip_broken_images(cleaned)
            cleaned = collapse_repetition(cleaned)
            if cleaned.strip():
                pages_text.append(cleaned)
        except Exception as e:
            print(f"   ⚠️  Could not read {fpath}: {e}")

    return "\n\n---\n\n".join(pages_text)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 6 ▸ CONFIGURATION  ← ★ ONLY CHANGE THIS ★
# ─────────────────────────────────────────────────────────────────

# ★ Enter the path to your PDF or image file here (.pdf, .jpg, .png, .bmp, .tiff, .webp):
PDF_PATH = "/content/Markup Alternatives.pdf"

# Optional settings:
DPI = 300          # Image resolution (higher = better quality, slower)
                   # Recommended: 200 (fast) | 300 (balanced) | 400 (high quality)

BATCH_SIZE = 5     # How many pages should be given to the GPU at once.
                   # On a T4 (15GB), giving 45+ pages at once causes CUDA OOM.
                    # If OOM still occurs, reduce this to 3 or 2.
                   # If you have an A100 (40GB), you can increase it to 10-15.

# ─────────────────────────────────────────────────────────────────

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 7 ▸ MAIN PROCESSING FUNCTION  (UPDATED — per-page gundam mode)
# ─────────────────────────────────────────────────────────────────

def process_pdf_to_markdown(pdf_path: str, dpi: int = 300, batch_size: int = None) -> str:
    """
    Take the PDF file and create a .md file with the same name.
    Now each page is processed SEPARATELY, in 'gundam' (crop) mode 
    demo-like accuracy, and the OOM/repetition risk is also lower.
    """
    pdf_path = Path(pdf_path).resolve()

    if not pdf_path.exists():
        raise FileNotFoundError(f"❌ PDF file was not found: {pdf_path}")
    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError(f"❌ File is not a PDF: {pdf_path.name}")

    md_path = pdf_path.with_suffix(".md")

    print("=" * 60)
    print("🚀 Unlimited-OCR PDF → Markdown Converter (gundam / per-page)")
    print("=" * 60)
    print(f"   📄 Input PDF  : {pdf_path.name}")
    print(f"   📝 Output MD  : {md_path.name}")
    print()

    print("📸 Step 1/3 — Converting PDF pages to images...")
    image_files = pdf_to_images(str(pdf_path), dpi=dpi)
    total = len(image_files)

    ocr_root_dir = tempfile.mkdtemp(prefix="ocr_results_")
    print(f"🔍 Step 2/3 — Running OCR on {total} pages (gundam mode, 1-by-1)...")
    print(f"   (It gives accuracy like the demo it may be a little slow)\n")

    page_dirs = []
    for idx, img_path in enumerate(image_files, start=1):
        page_dir = os.path.join(ocr_root_dir, f"page_{idx:04d}")
        os.makedirs(page_dir, exist_ok=True)

        print(f"   ▶️  Page {idx}/{total} ...", end=" ")
        try:
            model.infer(
                tokenizer,
                prompt='<image>document parsing.',
                image_file=img_path,
                output_path=page_dir,
                base_size=1024, image_size=640, crop_mode=True,   # ★ gundam mode
                max_length=32768,
                no_repeat_ngram_size=35, ngram_window=128,
                save_results=True,
            )
            print("✅")
        except torch.cuda.OutOfMemoryError:
            print("❌ OOM — retrying in 'base' mode (fallback)")
            torch.cuda.empty_cache()
            model.infer(
                tokenizer,
                prompt='<image>document parsing.',
                image_file=img_path,
                output_path=page_dir,
                base_size=1024, image_size=1024, crop_mode=False,  # base fallback
                max_length=32768,
                no_repeat_ngram_size=35, ngram_window=128,
                save_results=True,
            )
        finally:
            torch.cuda.empty_cache()

        page_dirs.append(page_dir)

    print("\n   ✅ OCR complete!")
    print(f"   📁 Raw output saved to: {ocr_root_dir}")

    print("\n📖 Step 3/3 — Reading & combining OCR output...")
    combined_parts = []
    for i, pdir in enumerate(page_dirs, start=1):
        part = read_and_combine_outputs(pdir, pdf_path.stem)
        combined_parts.append(part if part.strip() else f"*[Page {i} could not be read]*")

    combined_text = "\n\n---\n\n".join(combined_parts)

    final_md = f"""# {pdf_path.stem}

> *Generated by [Unlimited-OCR](https://github.com/baidu/Unlimited-OCR) (Baidu)*
> *Source PDF: `{pdf_path.name}`*

---

{combined_text}
"""

    with open(str(md_path), "w", encoding="utf-8") as f:
        f.write(final_md)

    char_count = len(combined_text)
    word_count = len(combined_text.split())
    md_size_kb = md_path.stat().st_size / 1024

    print()
    print("=" * 60)
    print("✅ SUCCESS!")
    print("=" * 60)
    print(f"   📝 Output file : {md_path}")
    print(f"   📊 Characters  : {char_count:,}")
    print(f"   📊 Words       : {word_count:,}")
    print(f"   💾 File size   : {md_size_kb:.1f} KB")

    try:
        img_tmp_dir = os.path.dirname(image_files[0])
        shutil.rmtree(img_tmp_dir, ignore_errors=True)
        print(f"   🧹 Temp images cleaned up.")
    except Exception:
        pass

    return str(md_path)


In [ ]:
# # ─────────────────────────────────────────────────────────────────
# # CELL 7 ▸ MAIN PROCESSING FUNCTION  (UPDATED — PDF + image formats)
# # ─────────────────────────────────────────────────────────────────

# # Unlimited-OCR (official README/infer.py) natively supports only:
# # • PDF          → converted to page images and then processed with OCR
# # • Single image → directly processed with OCR (jpg/jpeg/png/bmp/tiff/webp — any PIL-readable format)
# # Native support for docx/pptx/xlsx is NOT available — this is a pure image/PDF OCR model.

# SUPPORTED_IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif", ".webp"}
# SUPPORTED_EXTS = SUPPORTED_IMAGE_EXTS | {".pdf"}


# def process_document_to_markdown(input_path: str, dpi: int = 300, batch_size: int = None) -> str:
#     """
#     Take the PDF or a single image file and create a .md file with the same name.
#     - PDF   : each page is processed separately in 'gundam' mode
#     - Image : the file is directly processed in 'gundam' mode as a single page
#     """
#     input_path = Path(input_path).resolve()

#     if not input_path.exists():
#         raise FileNotFoundError(f"❌ File was not found: {input_path}")

#     ext = input_path.suffix.lower()
#     if ext not in SUPPORTED_EXTS:
#         raise ValueError(
#             f"❌ Unsupported file type: {ext}\n"
#             f"   Supported: {sorted(SUPPORTED_EXTS)}\n"
#             f"   (Unlimited-OCR sirf PDF aur image files support karta hai — "
#             f"docx/pptx/xlsx jaisa native support nahi hai)"
#         )

#     md_path = input_path.with_suffix(".md")

#     print("=" * 60)
#     print("🚀 Unlimited-OCR → Markdown Converter (gundam / per-page)")
#     print("=" * 60)
#     print(f"   📄 Input file : {input_path.name}")
#     print(f"   📝 Output MD  : {md_path.name}")
#     print()

#     # ── Step 1: input file → list of page images ────────────────────
#     cleanup_dir = None
#     if ext == ".pdf":
#         print("📸 Step 1/3 — Converting PDF pages to images...")
#         image_files = pdf_to_images(str(input_path), dpi=dpi)
#         cleanup_dir = os.path.dirname(image_files[0])
#     else:
#         print("📸 Step 1/3 — Single image file detected, PDF conversion skip...")
#         image_files = [str(input_path)]

#     total = len(image_files)

#     # ── Step 2: OCR (per-page, gundam mode, base-mode fallback) ─────
#     ocr_root_dir = tempfile.mkdtemp(prefix="ocr_results_")
#     print(f"🔍 Step 2/3 — Running OCR on {total} page(s) (gundam mode, 1-by-1)...")
#     print(f"   (It gives accuracy like the demo, though it may be a little slow.)\n")

#     page_dirs = []
#     for idx, img_path in enumerate(image_files, start=1):
#         page_dir = os.path.join(ocr_root_dir, f"page_{idx:04d}")
#         os.makedirs(page_dir, exist_ok=True)

#         print(f"   ▶️  Page {idx}/{total} ...", end=" ")
#         try:
#             model.infer(
#                 tokenizer,
#                 prompt='<image>document parsing.',
#                 image_file=img_path,
#                 output_path=page_dir,
#                 base_size=1024, image_size=640, crop_mode=True,   # gundam mode
#                 max_length=32768,
#                 no_repeat_ngram_size=35, ngram_window=128,
#                 save_results=True,
#             )
#             print("✅")
#         except torch.cuda.OutOfMemoryError:
#             print("❌ OOM — retrying in 'base' mode (fallback)")
#             torch.cuda.empty_cache()
#             model.infer(
#                 tokenizer,
#                 prompt='<image>document parsing.',
#                 image_file=img_path,
#                 output_path=page_dir,
#                 base_size=1024, image_size=1024, crop_mode=False,  # base fallback
#                 max_length=32768,
#                 no_repeat_ngram_size=35, ngram_window=128,
#                 save_results=True,
#             )
#         finally:
#             torch.cuda.empty_cache()

#         page_dirs.append(page_dir)

#     print("\n   ✅ OCR complete!")
#     print(f"   📁 Raw output saved to: {ocr_root_dir}")

#     # ── Step 3: combine ───────────────────────────────────────────
#     print("\n📖 Step 3/3 — Reading & combining OCR output...")
#     combined_parts = []
#     for i, pdir in enumerate(page_dirs, start=1):
#         part = read_and_combine_outputs(pdir, input_path.stem)
#         combined_parts.append(part if part.strip() else f"*[Page {i} could not be read]*")

#     combined_text = "\n\n---\n\n".join(combined_parts)

#     final_md = f"""# {input_path.stem}

# > *Generated by [Unlimited-OCR](https://github.com/baidu/Unlimited-OCR) (Baidu)*
# > *Source file: `{input_path.name}`*

# ---

# {combined_text}
# """

#     with open(str(md_path), "w", encoding="utf-8") as f:
#         f.write(final_md)

#     char_count = len(combined_text)
#     word_count = len(combined_text.split())
#     md_size_kb = md_path.stat().st_size / 1024

#     print()
#     print("=" * 60)
#     print("✅ SUCCESS!")
#     print("=" * 60)
#     print(f"   📝 Output file : {md_path}")
#     print(f"   📊 Characters  : {char_count:,}")
#     print(f"   📊 Words       : {word_count:,}")
#     print(f"   💾 File size   : {md_size_kb:.1f} KB")

#     if cleanup_dir:
#         try:
#             shutil.rmtree(cleanup_dir, ignore_errors=True)
#             print(f"   🧹 Temp images cleaned up.")
#         except Exception:
#             pass

#     return str(md_path)


# # ★ Backward-compatible alias — There’s no need to change anything in Cell 8
# process_pdf_to_markdown = process_document_to_markdown


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 8 ▸ RUN!
# ─────────────────────────────────────────────────────────────────

output_md_path = process_pdf_to_markdown(PDF_PATH, dpi=DPI, batch_size=BATCH_SIZE)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 9 ▸ PREVIEW + DOWNLOAD
# ─────────────────────────────────────────────────────────────────

# Preview the first 2000 characters of the output
print("\n" + "═" * 60)
print("📖 MARKDOWN PREVIEW (first 2000 chars)")
print("═" * 60)
with open(output_md_path, "r", encoding="utf-8") as f:
    preview = f.read(2000)
print(preview)
if len(open(output_md_path).read()) > 2000:
    print("\n... [truncated — full content in downloaded file] ...")

# Auto-download in Colab
try:
    from google.colab import files
    print(f"\n⬇️  Downloading: {Path(output_md_path).name}")
    files.download(output_md_path)
    print("✅ Download started!")
except ImportError:
    # It is running outside Colab
    print(f"\n📁 File saved at: {output_md_path}")
    print("   (Colab nahi hai — manually copy karo)")
except Exception as e:
    print(f"\n⚠️  Auto-download failed: {e}")
    print(f"   Manually download it: {output_md_path}")
    print("   Or find it in the Colab left sidebar.")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 10 ▸ BATCH MODE (optional — multiple PDFs at once)
# ─────────────────────────────────────────────────────────────────

def process_multiple_pdfs(pdf_folder: str, dpi: int = 300, batch_size: int = 5):
    """
    Process all PDF files in a folder.

    Usage:
        process_multiple_pdfs("/content/my_pdfs/")
    """
    pdf_files = list(Path(pdf_folder).glob("*.pdf"))
    if not pdf_files:
        print(f"❌ No PDF was found: {pdf_folder}")
        return

    print(f"📚 {len(pdf_files)} PDFs found. Processing...")
    results = []

    for i, pdf in enumerate(pdf_files, 1):
        print(f"\n[{i}/{len(pdf_files)}] Processing: {pdf.name}")
        try:
            md = process_pdf_to_markdown(str(pdf), dpi=dpi, batch_size=batch_size)
            results.append({"pdf": pdf.name, "md": md, "status": "✅ Success"})
        except Exception as e:
            print(f"   ❌ Failed: {e}")
            results.append({"pdf": pdf.name, "md": None, "status": f"❌ {e}"})

    print("\n" + "=" * 60)
    print("📊 BATCH SUMMARY")
    print("=" * 60)
    for r in results:
        print(f"  {r['status']}  {r['pdf']}")
    print("=" * 60)

    # Batch download (Colab only)
    try:
        from google.colab import files
        for r in results:
            if r["md"]:
                files.download(r["md"])
    except ImportError:
        pass

    return results

# If you want to use batch mode:
# process_multiple_pdfs("/content/pdfs/")